# MovieLens 1M data preparation

This notebook prepares MovieLens 1M for content-based recommendation using the exact shared protocol from Lijie's model-based transfer.

It stops before movie vectors or recommendations are created.

## Pipeline

```text
ratings.dat + movies.dat
          ↓
Validate ratings and movie metadata
          ↓
Ratings 4–5 become positives
          ↓
Iterative positive 5-core
          ↓
One validation and one test positive per user
          ↓
One positive + 99 unseen candidates
          ↓
Join all warm movies to metadata
          ↓
Ready for content feature construction
```

Ratings 1–3 remain observed interactions. They are excluded from distractor sampling but do not build positive user profiles.

In [ ]:
from __future__ import annotations

import os
import sys
from pathlib import Path

import pandas as pd

def find_repo_root(start: Path = Path.cwd()) -> Path:
    for path in (start, *start.parents):
        if (path / "src" / "data.py").exists():
            return path
    raise FileNotFoundError("Run this notebook inside the dam repository.")

REPO_ROOT = find_repo_root()
PROJECT_DIR = REPO_ROOT / "content_collab" / "movielens_dataset"
for path in (REPO_ROOT, PROJECT_DIR):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

from movielens_content import (
    assert_archived_counts,
    prepare_shared_data,
    resolve_dataset_dir,
)

DATASET_DIR = resolve_dataset_dir(
    os.getenv("MOVIELENS_1M_DIR"),
    REPO_ROOT,
    auto_download=True,
)
print("Dataset:", DATASET_DIR)

## 1. Load and verify the shared protocol

Expected full-data counts:

| Quantity | Expected |
| --- | ---: |
| Raw ratings | 1,000,209 |
| Positive-core users | 6,034 |
| Warm movies | 3,125 |
| Training positives | 562,308 |
| Validation positives | 6,034 |
| Test positives | 6,034 |
| Candidates per split | 603,400 |

In [ ]:
shared = prepare_shared_data(DATASET_DIR)
assert_archived_counts(shared)

summary = pd.Series({
    "raw ratings": len(shared.ratings_raw),
    "ambiguous duplicate pairs": shared.ambiguous_pair_count,
    "positive rows before 5-core": shared.core_stats.input_positive_count,
    "positive rows after 5-core": shared.core_stats.output_positive_count,
    "eligible users": len(shared.eligible_users),
    "warm movies": len(shared.warm_items),
    "training positives": len(shared.split.train),
    "validation positives": len(shared.split.validation),
    "test positives": len(shared.split.test),
    "validation candidates": len(shared.validation_candidates),
    "test candidates": len(shared.test_candidates),
})
summary_display = summary.to_frame("count")
summary_display["count"] = summary_display["count"].map(lambda value: f"{int(value):,}")
display(summary_display)
print("✅ Archived protocol counts match.")

## 2. Rating interpretation

```text
rating 4–5 → positive
rating 1–3 → observed, but not positive
```

Low ratings are not sampled as unseen movies for that user.

In [ ]:
rating_summary = (
    shared.ratings_raw["rating"]
    .value_counts()
    .sort_index()
    .rename_axis("rating")
    .reset_index(name="rows")
)
rating_summary["role"] = rating_summary["rating"].map(
    lambda rating: "positive" if rating >= 4 else "observed, not positive"
)
display(rating_summary)
assert shared.ratings_raw["rating"].between(1, 5).all()
assert shared.ambiguous_pair_count == 0

## 3. Inspect the per-user split

Only training positives may build a user profile. Validation chooses the content configuration. Test is evaluated after selection.

In [ ]:
example_user = shared.eligible_users[0]
parts = []
for name, frame in [
    ("training", shared.split.train),
    ("validation", shared.split.validation),
    ("test", shared.split.test),
]:
    current = frame.loc[frame["user_id"].eq(example_user)].copy()
    current["partition"] = name
    parts.append(current)

example_split = pd.concat(parts, ignore_index=True).merge(
    shared.metadata[["movie_id", "title_with_year"]],
    left_on="anime_id",
    right_on="movie_id",
    how="left",
)
display(
    example_split[
        ["partition", "user_id", "anime_id", "title_with_year", "rating"]
    ].sort_values(["partition", "rating"], ascending=[True, False])
)

## 4. Inspect fixed validation candidates

Each evaluated user receives one hidden positive and 99 warm movies with no observed rating from that user.

In [ ]:
example_candidates = shared.validation_candidates.loc[
    shared.validation_candidates["user_id"].eq(example_user)
].merge(
    shared.metadata[["movie_id", "title_with_year"]],
    left_on="item_id",
    right_on="movie_id",
    how="left",
)
display(example_candidates.head(12))
print("Positive rows:", int(example_candidates["label"].sum()))
print("Distractor rows:", int(example_candidates["label"].eq(0).sum()))
assert len(example_candidates) == 100
assert int(example_candidates["label"].sum()) == 1

## 5. Movie metadata preparation

The year is removed from the title and represented separately as a decade.

Example:

```text
Toy Story (1995)
→ title: Toy Story
→ decade: 1990s
→ genres: Animation | Children's | Comedy
```

In [ ]:
display(
    shared.metadata[
        [
            "movie_id",
            "title_with_year",
            "title",
            "release_year",
            "release_decade",
            "genres",
        ]
    ].head(10)
)

coverage = pd.Series({
    "warm movie rows": len(shared.metadata),
    "missing titles": shared.metadata["title"].isna().sum(),
    "missing genres": shared.metadata["genres"].isna().sum(),
    "missing release years": shared.metadata["release_year"].isna().sum(),
    "distinct genres": len({
        genre
        for value in shared.metadata["genres"]
        for genre in value.split("|")
        if genre != "(no genres listed)"
    }),
    "distinct decades": shared.metadata["release_decade"].nunique(),
})
display(coverage.to_frame("count"))

assert len(shared.metadata) == 3_125
assert set(shared.metadata["movie_id"]) == set(shared.warm_items)
assert shared.metadata[["movie_id", "title", "genres"]].notna().all().all()
print("✅ Every warm movie has usable metadata.")

## 6. Prepared outputs

The next notebook receives:

| Object | Purpose |
| --- | --- |
| `shared.split.train` | Build rating-weighted user profiles |
| `shared.validation_candidates` | Select M1–M4 |
| `shared.test_candidates` | Final evaluation once |
| `shared.observed_by_user` | Verify distractors are unobserved |
| `shared.metadata` | Construct genre, title, and decade vectors |
| `shared.eligible_users` | Fixed user universe |
| `shared.warm_items` | Fixed movie universe |

No recommendation scores have been calculated at this point.